In [ ]:
import pathlib
import duckdb
import pandas as pd
import ipywidgets as w
from IPython.display import display, HTML
from dotenv import load_dotenv

from irp.features.metrics import compute_metrics

_ROOT = pathlib.Path().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent

load_dotenv()
con = duckdb.connect(str(_ROOT / "data/irp.duckdb"), read_only=True)
con.execute("PRAGMA disable_progress_bar")

fundamentals = {t: con.execute(f"SELECT * FROM {t}").df() for t in ("income","balance","cashflow")}
# Restrict prices to tickers that have fundamentals (screener needs both)
prices = con.execute(
    'SELECT ticker, date, open, high, low, close, volume FROM prices '
    'WHERE ticker IN (SELECT DISTINCT "Ticker" FROM income)'
).df()

metrics_cache = {}
def get_metrics(period: str) -> pd.DataFrame:
    if period not in metrics_cache:
        metrics_cache[period] = compute_metrics(fundamentals, prices, period, latest=True)
    return metrics_cache[period]

_warm = get_metrics("annual")
print(f"Loaded {_warm.shape[0]} tickers, {_warm.shape[1]} metric columns")

In [ ]:
# (column name, display format) — drives both UI dropdown and table rendering
METRIC_DEFS = [
    # valuation
    ("P/E", "ratio"),
    ("Earnings Yield", "pct"),
    ("P/B", "ratio"),
    ("P/S", "ratio"),
    ("P/FCF", "ratio"),
    ("EV/EBITDA", "ratio"),
    ("EV/Sales", "ratio"),
    # profitability
    ("Gross Margin", "pct"),
    ("Operating Margin", "pct"),
    ("Net Margin", "pct"),
    ("ROE", "pct"),
    ("ROA", "pct"),
    ("ROIC", "pct"),
    # growth
    ("Revenue Growth YoY", "pct"),
    ("Revenue 3Y CAGR", "pct"),
    ("EPS Growth YoY", "pct"),
    ("EPS 3Y CAGR", "pct"),
    ("FCF Growth YoY", "pct"),
    ("FCF 3Y CAGR", "pct"),
    ("OpInc Growth YoY", "pct"),
    ("OpInc 3Y CAGR", "pct"),
    # health
    ("Debt/Equity", "ratio"),
    ("Net Debt/EBITDA", "ratio"),
    ("Current Ratio", "ratio"),
    ("Quick Ratio", "ratio"),
    ("Interest Coverage", "ratio"),
    # cash flow
    ("FCF Yield", "pct"),
    ("FCF Margin", "pct"),
    ("Cash Conversion", "ratio"),
    # dividend
    ("Dividend Yield", "pct"),
    ("Payout Ratio", "pct"),
    # momentum
    ("Return 1M", "pct"),
    ("Return 3M", "pct"),
    ("Return 6M", "pct"),
    ("Return 1Y", "pct"),
    ("Return YTD", "pct"),
    ("Dist from 52w High", "pct"),
    ("Dist from 52w Low", "pct"),
    ("Volatility (annualized)", "pct"),
    # scores
    ("Piotroski F", "int"),
    ("Altman Z", "ratio"),
]
METRIC_NAMES = [m[0] for m in METRIC_DEFS]
METRIC_FMT = dict(METRIC_DEFS)

DEFAULT_COLS = [
    "P/E",
    "P/B",
    "ROE",
    "Net Margin",
    "Revenue 3Y CAGR",
    "Debt/Equity",
    "FCF Yield",
    "Dividend Yield",
    "Return 1Y",
    "Piotroski F",
]


def _fmt_cell(v, kind):
    if pd.isna(v):
        return ""
    if kind == "pct":
        return f"{v:,.1%}"
    if kind == "ratio":
        return f"{v:,.2f}"
    if kind == "int":
        return f"{int(v)}"
    return str(v)


_STYLE = """<style>
table.scr { border-collapse: collapse; font-size: 12px; color: #e0e0e0; background: #141414; }
table.scr th { text-align: right; padding: 4px 8px; border-bottom: 1px solid #555;
               position: sticky; top: 0; background: #1e1e1e; color: #e0e0e0; }
table.scr td { text-align: right; padding: 3px 8px; white-space: nowrap;
               background: #141414; color: #e0e0e0; }
table.scr tr:nth-child(even) td { background: #1a1a1a; }
table.scr td:first-child, table.scr th:first-child { text-align: left; font-weight: bold; }
table.scr tr:hover td { background: #2a2a2a; color: #e0e0e0; }
</style>"""


# _STYLE = """<style>
# table.scr { 
#   border-collapse: collapse; 
#   font-size: 12px; 
#   color: #d0d0d0; 
#   background: #121212; 
# }

# table.scr th { 
#   text-align: right; 
#   padding: 4px 8px; 
#   border-bottom: 1px solid #444;
#   position: sticky; 
#   top: 0; 
#   background: #1f1f1f; 
#   color: #e6e6e6; 
# }

# table.scr td { 
#   text-align: right; 
#   padding: 3px 8px; 
#   white-space: nowrap;
#   background: #121212; 
# }

# table.scr tr:nth-child(even) td { 
#   background: #1e1e1e;   /* stronger contrast */
# }

# table.scr tr:hover td { 
#   background: #ff2f2f;   /* clear hover */
# }
# </style>"""

In [ ]:
# --- top controls ---
period_dd = w.Dropdown(
    options=[("Annual", "annual"), ("TTM", "ttm")],
    value="annual",
    description="Period:",
    style={"description_width": "initial"},
)

cols_sel = w.SelectMultiple(
    options=METRIC_NAMES,
    value=tuple(DEFAULT_COLS),
    description="Show columns:",
    rows=10,
    layout=w.Layout(width="320px"),
    style={"description_width": "initial"},
)

sort_dd = w.Dropdown(
    options=METRIC_NAMES,
    value="P/E",
    description="Sort by:",
    style={"description_width": "initial"},
)
sort_asc = w.Checkbox(value=True, description="Ascending")
limit_in = w.IntText(
    value=50,
    description="Limit:",
    layout=w.Layout(width="140px"),
    style={"description_width": "initial"},
)

# --- dynamic filter list ---
filter_rows: list[
    dict
] = []  # each: {"box": HBox, "metric": Dropdown, "min": FloatText, "max": FloatText}
filters_box = w.VBox([])


def _add_filter(
    _=None, metric: str = None, vmin: float | None = None, vmax: float | None = None
):
    metric_dd = w.Dropdown(
        options=METRIC_NAMES,
        value=metric or METRIC_NAMES[0],
        layout=w.Layout(width="240px"),
    )
    min_in = w.FloatText(
        value=vmin if vmin is not None else None,
        description="min",
        layout=w.Layout(width="160px"),
        style={"description_width": "initial"},
    )
    max_in = w.FloatText(
        value=vmax if vmax is not None else None,
        description="max",
        layout=w.Layout(width="160px"),
        style={"description_width": "initial"},
    )
    rm_btn = w.Button(
        description="x", layout=w.Layout(width="32px"), button_style="warning"
    )
    row = w.HBox([metric_dd, min_in, max_in, rm_btn])
    entry = {"box": row, "metric": metric_dd, "min": min_in, "max": max_in}
    filter_rows.append(entry)
    rm_btn.on_click(lambda _b: _remove_filter(entry))
    filters_box.children = tuple(e["box"] for e in filter_rows)


def _remove_filter(entry):
    filter_rows.remove(entry)
    filters_box.children = tuple(e["box"] for e in filter_rows)


add_btn = w.Button(description="+ Add filter", button_style="info")
add_btn.on_click(_add_filter)

apply_btn = w.Button(description="Apply", button_style="primary")
out = w.Output()


def _apply(_=None):
    out.clear_output(wait=True)
    with out:
        try:
            df = get_metrics(period_dd.value).copy()
            n0 = len(df)
            for e in filter_rows:
                col = e["metric"].value
                lo, hi = e["min"].value, e["max"].value
                if not pd.isna(lo):
                    df = df[df[col] >= lo]
                if not pd.isna(hi):
                    df = df[df[col] <= hi]
            df = df.sort_values(
                sort_dd.value, ascending=sort_asc.value, na_position="last"
            )
            df = df.head(limit_in.value)
            cols = ["ticker", "period"] + list(cols_sel.value)
            view = df[cols].copy()
            for c in cols_sel.value:
                view[c] = view[c].apply(lambda v, k=METRIC_FMT[c]: _fmt_cell(v, k))
            print(f"{len(df)} of {n0} tickers match")
            display(
                HTML(
                    _STYLE
                    + '<div style="max-height: 500px; overflow-y: auto; border: 1px solid #333;">'
                    + view.to_html(index=False, classes="scr", escape=False)
                    + '</div>'
                )
            )
        except Exception as e:
            print(f"Error: {e}")


apply_btn.on_click(_apply)

# Seed with two example filters
_add_filter(metric="P/E", vmin=None, vmax=20.0)
_add_filter(metric="ROE", vmin=0.15, vmax=None)

display(
    w.HBox([w.VBox([period_dd, sort_dd, sort_asc, limit_in]), cols_sel]),
    w.Label("Filters (AND-joined):"),
    filters_box,
    w.HBox([add_btn, apply_btn]),
    out,
)

Improvements:
- filter on industries
- clicking a row adds the ticker to a selection list.
- market cap, enterprice value
- peg ratio

